# Evaluate one-output spot separation models

This notebook matches the current `train.py` one-output setup:

- the U-Net has `n_classes=1`
- logits are converted to the first spot intensity via `sigmoid(logits) * image`
- the second spot is a residual: `image - pred_spot_1`
- target channels are remapped per sample so the direct output is compared to the better matching ground-truth spot

All outputs are written to the folder specified by `OUTPUT_DIR` at the top.

In [ ]:
from pathlib import Path
import json
import re
import time

import h5py
import hdf5plugin  # noqa: F401 - registers compressed HDF5 filters
import matplotlib.pyplot as plt
import numpy as np
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("This notebook needs pandas for result tables. Install it with: uv add pandas") from exc
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from unet_model import UNet

## Configuration

In [ ]:
DATA_PATH = Path("data_10000_spots/augmented_spots_train.h5")
OUTPUT_DIR = Path("evaluation/current_oneoutput")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_SIZE = 5000
RANDOM_SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0
IMG_SCALE = 1.0

FOREGROUND_THRESHOLD = 1e-4
PREDICTION_MASK_THRESHOLD = 1e-4
DICE_WEIGHT = 0.5
RMSE_WEIGHT = 0.5

AMP = torch.cuda.is_available()
BILINEAR = False  # must match the training flag used for the checkpoint

# Option 1: fill this manually, e.g. [Path("serverdaten/oneoutput/final_model.pth"), ...]
MODEL_PATHS = []

# Option 2: automatic local search if MODEL_PATHS is empty.
MODEL_SEARCH_ROOTS = [Path("checkpoints"), Path("serverdaten"), Path("output")]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## Inspect HDF5 structure

In [ ]:
def sample_group_names(h5_file):
    return sorted(
        name
        for name, obj in h5_file.items()
        if isinstance(obj, h5py.Group) and "image" in obj and "spot_images" in obj
    )


def inspect_h5_metadata(path):
    if not path.exists():
        raise FileNotFoundError(f"HDF5 file not found: {path.resolve()}")
    with h5py.File(path, "r") as f:
        names = sample_group_names(f)
        if not names:
            raise ValueError("No sample groups with image and spot_images were found.")
        first = f[names[0]]
        return {
            "n_samples": len(names),
            "root_attrs": dict(f.attrs),
            "first_sample": names[0],
            "first_sample_attrs": dict(first.attrs),
            "datasets": {key: {"shape": value.shape, "dtype": str(value.dtype)} for key, value in first.items()},
            "has_spot_masks": "spot_masks" in first,
            "has_sample_donors_attr": "donors" in first.attrs,
            "has_root_provenance": "provenance" in f,
            "provenance_keys": list(f["provenance"].keys()) if "provenance" in f else [],
        }


report = inspect_h5_metadata(DATA_PATH)
report

In [ ]:
def choose_subset_indices(n_samples, subset_size=SUBSET_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    size = min(subset_size, n_samples)
    return np.sort(rng.choice(n_samples, size=size, replace=False))


with h5py.File(DATA_PATH, "r") as f:
    ALL_SAMPLE_NAMES = sample_group_names(f)

SUBSET_INDICES = choose_subset_indices(len(ALL_SAMPLE_NAMES))
SUBSET_NAMES = [ALL_SAMPLE_NAMES[i] for i in SUBSET_INDICES]
print(f"Using {len(SUBSET_NAMES)} random samples from {len(ALL_SAMPLE_NAMES)} total samples.")
print(SUBSET_NAMES[:5])

## Dataset loader

In [ ]:
def normalize_image_and_targets(image, targets):
    image = image.astype(np.float32, copy=False)
    targets = targets.astype(np.float32, copy=False)
    finite_image = np.isfinite(image)
    if not finite_image.any():
        zeros_image = np.zeros_like(image, dtype=np.float32)
        zeros_targets = np.zeros_like(targets, dtype=np.float32)
        return zeros_image, zeros_targets, zeros_targets.copy(), np.float32(1.0)

    clean_image = np.where(finite_image, image, 0.0)
    clean_targets = np.where(np.isfinite(targets), targets, 0.0)
    np.maximum(clean_targets, 0.0, out=clean_targets)

    finite_values = clean_image[finite_image]
    high = float(np.percentile(finite_values, 99.9))
    if high <= 0:
        high = float(finite_values.max())
    if high <= 0:
        zeros_image = np.zeros_like(image, dtype=np.float32)
        zeros_targets = np.zeros_like(targets, dtype=np.float32)
        return zeros_image, zeros_targets, zeros_targets.copy(), np.float32(1.0)

    positive_values = clean_image[clean_image > 0]
    positive_high = float(np.percentile(positive_values, 99.9)) if positive_values.size else high
    if positive_high <= 0:
        positive_high = high

    normalized_targets = np.clip(clean_targets, 0.0, high) / high
    normalized_targets_positive = np.clip(clean_targets, 0.0, positive_high) / positive_high
    normalized_image = normalized_targets.sum(axis=0, dtype=np.float32)
    positive_scale = np.float32(high / positive_high)
    return (
        normalized_image.astype(np.float32, copy=False),
        normalized_targets.astype(np.float32, copy=False),
        normalized_targets_positive.astype(np.float32, copy=False),
        positive_scale,
    )

def sample_index_from_name(name):
    match = re.search(r"(\d+)$", name)
    return int(match.group(1)) if match else None


def decode_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return value


def donor_keys_from_sample(h5_file, sample_name):
    group = h5_file[sample_name]
    if "donors" in group.attrs:
        donors = json.loads(decode_attr(group.attrs["donors"]))
        return [
            "|".join(str(donor.get(key, "?")) for key in ("experiment", "source_file", "donor_group", "source_row", "blob_id"))
            for donor in donors
        ]
    if "provenance" in h5_file and "source_donor_groups" in h5_file["provenance"]:
        sample_index = sample_index_from_name(sample_name)
        if sample_index is not None:
            provenance = h5_file["provenance"]
            donor_groups = provenance["source_donor_groups"][sample_index]
            rows = provenance["source_rows"][sample_index] if "source_rows" in provenance else ["?", "?"]
            blob_ids = provenance["source_blob_ids"][sample_index] if "source_blob_ids" in provenance else ["?", "?"]
            experiment = decode_attr(group.attrs.get("experiment", "?"))
            return [f"{experiment}|donor_group={int(d)}|row={int(r)}|blob={int(b)}" for d, r, b in zip(donor_groups, rows, blob_ids)]
    return []


class H5OneOutputSubsetDataset(Dataset):
    def __init__(self, h5_path, sample_names, img_scale=IMG_SCALE):
        self.h5_path = Path(h5_path)
        self.sample_names = list(sample_names)
        self.img_scale = img_scale

    def __len__(self):
        return len(self.sample_names)

    def __getitem__(self, idx):
        sample_name = self.sample_names[idx]
        with h5py.File(self.h5_path, "r") as f:
            group = f[sample_name]
            image = group["image"][()]
            targets = group["spot_images"][()]
            masks = group["spot_masks"][()] if "spot_masks" in group else None
            donor_keys = donor_keys_from_sample(f, sample_name)
            attrs = {key: decode_attr(value) for key, value in group.attrs.items() if key != "donors"}

        if image.ndim == 3:
            image = image.mean(axis=-1)
        if targets.ndim != 3 or targets.shape[0] != 2:
            raise ValueError(f"{sample_name}: expected spot_images shape (2, H, W), got {targets.shape}")
        image, targets, positive_targets, positive_scale = normalize_image_and_targets(image, targets)
        if masks is None:
            masks = targets > FOREGROUND_THRESHOLD
        else:
            masks = masks.astype(bool, copy=False)

        image_tensor = torch.from_numpy(image).unsqueeze(0)
        target_tensor = torch.from_numpy(targets)
        target_positive_tensor = torch.from_numpy(positive_targets)
        positive_scale_tensor = torch.tensor(positive_scale, dtype=torch.float32)
        mask_tensor = torch.from_numpy(masks.astype(bool, copy=False))

        if self.img_scale != 1.0:
            size = (
                max(1, int(image_tensor.shape[1] * self.img_scale)),
                max(1, int(image_tensor.shape[2] * self.img_scale)),
            )
            image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False).squeeze(0)
            target_tensor = F.interpolate(target_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False).squeeze(0)
            target_positive_tensor = F.interpolate(target_positive_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False).squeeze(0)
            mask_tensor = F.interpolate(mask_tensor.float().unsqueeze(0), size=size, mode="nearest").squeeze(0).bool()

        return {
            "image": image_tensor,
            "target_intensity": target_tensor,
            "target_intensity_positive": target_positive_tensor,
            "positive_scale": positive_scale_tensor,
            "target_mask": mask_tensor,
            "sample_name": sample_name,
            "donor_keys": donor_keys,
            "attrs": attrs,
        }


def collate_batch(batch):
    return {
        "image": torch.stack([item["image"] for item in batch]),
        "target_intensity": torch.stack([item["target_intensity"] for item in batch]),
        "target_intensity_positive": torch.stack([item["target_intensity_positive"] for item in batch]),
        "positive_scale": torch.stack([item["positive_scale"] for item in batch]),
        "target_mask": torch.stack([item["target_mask"] for item in batch]),
        "sample_name": [item["sample_name"] for item in batch],
        "donor_keys": [item["donor_keys"] for item in batch],
        "attrs": [item["attrs"] for item in batch],
    }


dataset = H5OneOutputSubsetDataset(DATA_PATH, SUBSET_NAMES)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_batch)

## One-output prediction mapping and metrics

In [ ]:
EPS = 1e-7
SPOT_BIN_ORDER = ["<50%", "50-70%", "70-90%", ">90%"]


def first_spot_prediction(logits, image):
    if logits.shape[1] != 1:
        raise ValueError(f"Expected one output channel for current one-output model, got {logits.shape[1]}")
    return torch.sigmoid(logits) * image


def compose_spot_predictions(first_spot, image):
    second_spot = (image - first_spot).clamp_min(0.0)
    return torch.cat((first_spot, second_spot), dim=1)


def soft_dice(prediction, target):
    numerator = 2.0 * (prediction * target).sum(dim=(2, 3)) + EPS
    denominator = prediction.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + EPS
    return numerator / denominator


def single_spot_cost(first_spot, candidate_target):
    dice_loss = 1.0 - soft_dice(first_spot, candidate_target).mean(dim=1)
    error = (first_spot - candidate_target).abs()
    foreground = (candidate_target > FOREGROUND_THRESHOLD).float()
    weights = 1.0 + 8.0 * foreground
    intensity = (error * weights).sum(dim=(1, 2, 3)) / weights.sum(dim=(1, 2, 3)).clamp_min(1.0)
    target_background = (candidate_target <= FOREGROUND_THRESHOLD).float()
    background = (first_spot.abs() * target_background).sum(dim=(1, 2, 3)) / target_background.sum(dim=(1, 2, 3)).clamp_min(1.0)
    return dice_loss + 0.25 * intensity + 0.01 * background


def align_targets_for_first_output(first_spot, target_intensity, target_mask):
    first_cost = single_spot_cost(first_spot, target_intensity[:, 0:1])
    second_cost = single_spot_cost(first_spot, target_intensity[:, 1:2])
    use_second = second_cost < first_cost
    aligned_intensity = target_intensity.clone()
    aligned_mask = target_mask.clone()
    aligned_intensity[use_second] = target_intensity[use_second].flip(1)
    aligned_mask[use_second] = target_mask[use_second].flip(1)
    return aligned_intensity, aligned_mask, use_second


def prediction_masks(prediction):
    return prediction > PREDICTION_MASK_THRESHOLD


def mask_dice_iou(pred_mask, target_mask):
    intersection = (pred_mask & target_mask).sum(dim=(2, 3)).float()
    pred_area = pred_mask.sum(dim=(2, 3)).float()
    target_area = target_mask.sum(dim=(2, 3)).float()
    union = (pred_mask | target_mask).sum(dim=(2, 3)).float()
    dice = (2.0 * intersection + EPS) / (pred_area + target_area + EPS)
    iou = (intersection + EPS) / (union + EPS)
    return dice, iou


def foreground_rmse(prediction, target_intensity, target_mask):
    weights = target_mask.float()
    squared_error = (prediction - target_intensity).square() * weights
    pixel_count = weights.sum(dim=(2, 3)).clamp_min(1.0)
    rmse = torch.sqrt(squared_error.sum(dim=(2, 3)) / pixel_count)
    target_rms = torch.sqrt((target_intensity.square() * weights).sum(dim=(2, 3)) / pixel_count).clamp_min(EPS)
    nrmse = rmse / target_rms
    intensity_score = (1.0 - nrmse).clamp(0.0, 1.0)
    return rmse, nrmse, intensity_score


def soft_intensity_dice(prediction, target_intensity, target_mask):
    weights = target_mask.float()
    pred = prediction * weights
    target = target_intensity * weights
    numerator = 2.0 * (pred * target).sum(dim=(2, 3)) + EPS
    denominator = pred.square().sum(dim=(2, 3)) + target.square().sum(dim=(2, 3)) + EPS
    return numerator / denominator


def integrated_intensity_error(prediction, target_intensity, target_mask):
    weights = target_mask.float()
    pred_integral = (prediction * weights).sum(dim=(2, 3))
    target_integral = (target_intensity * weights).sum(dim=(2, 3))
    signed_error = pred_integral - target_integral
    absolute_error = signed_error.abs()
    relative_error = signed_error / target_integral.clamp_min(EPS)
    absolute_relative_error = absolute_error / target_integral.clamp_min(EPS)
    return pred_integral, target_integral, signed_error, absolute_error, relative_error, absolute_relative_error


def spot_bin(score):
    if score < 0.50:
        return "<50%"
    if score < 0.70:
        return "50-70%"
    if score < 0.90:
        return "70-90%"
    return ">90%"


def compute_metrics(prediction, target_intensity, target_mask, target_intensity_positive=None, positive_scale=None):
    pred_mask = prediction_masks(prediction)
    dice, iou = mask_dice_iou(pred_mask, target_mask)
    rmse, nrmse, _ = foreground_rmse(prediction, target_intensity, target_mask)
    soft_i_dice = soft_intensity_dice(prediction, target_intensity, target_mask)
    rmse_whole = torch.sqrt((prediction - target_intensity).square().mean(dim=(2, 3)))
    target_rms_whole = torch.sqrt(target_intensity.square().mean(dim=(2, 3))).clamp_min(EPS)
    nrmse_whole = rmse_whole / target_rms_whole
    soft_i_dice_whole = (
        2.0 * (prediction * target_intensity).sum(dim=(2, 3)) + EPS
    ) / (prediction.square().sum(dim=(2, 3)) + target_intensity.square().sum(dim=(2, 3)) + EPS)
    pred_integral, target_integral, iie, iie_abs, iie_rel, iie_abs_rel = integrated_intensity_error(
        prediction, target_intensity, target_mask
    )

    if target_intensity_positive is not None:
        prediction_for_positive = prediction
        if positive_scale is not None:
            prediction_for_positive = prediction * positive_scale.view(-1, 1, 1, 1)
        _, nrmse_positive, intensity_score = foreground_rmse(
            prediction_for_positive, target_intensity_positive, target_mask
        )
    else:
        nrmse_positive = nrmse
        intensity_score = (1.0 - nrmse).clamp(0.0, 1.0)

    recognition = (DICE_WEIGHT * dice + RMSE_WEIGHT * intensity_score).clamp(0.0, 1.0)
    return {
        "dice": dice,
        "iou": iou,
        "rmse": rmse,
        "nrmse": nrmse,
        "rmse_whole": rmse_whole,
        "nrmse_whole": nrmse_whole,
        "nrmse_positive": nrmse_positive,
        "intensity_score": intensity_score,
        "soft_intensity_dice": soft_i_dice,
        "soft_intensity_dice_whole": soft_i_dice_whole,
        "pred_integral": pred_integral,
        "target_integral": target_integral,
        "integrated_intensity_error": iie,
        "integrated_intensity_abs_error": iie_abs,
        "integrated_intensity_relative_error": iie_rel,
        "integrated_intensity_abs_relative_error": iie_abs_rel,
        "spot_recognition": recognition,
    }

## Model loading

In [ ]:
def discover_model_paths():
    if MODEL_PATHS:
        return [Path(path) for path in MODEL_PATHS]
    paths = []
    for root in MODEL_SEARCH_ROOTS:
        if root.exists():
            paths.extend(root.rglob("*.pth"))
            paths.extend(root.rglob("*.pt"))
    return sorted(set(paths))


def load_oneoutput_model(path):
    model = UNet(n_channels=1, n_classes=1, bilinear=BILINEAR).to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict):
        checkpoint.pop("mask_values", None)
    model.load_state_dict(checkpoint)
    model.eval()
    return model


model_paths = discover_model_paths()
print(f"Found {len(model_paths)} model file(s).")
for path in model_paths:
    print(path)

## Evaluation loop

In [ ]:
def evaluate_model(model_path):
    model = load_oneoutput_model(model_path)
    sample_rows = []
    spot_rows = []
    n_batches = 0
    n_samples = 0
    n_spots = 0
    inference_seconds = 0.0
    metric_seconds = 0.0
    model_start = time.perf_counter()

    with torch.no_grad():
        for batch in tqdm(loader, desc=model_path.name):
            batch_start = time.perf_counter()
            n_batches += 1
            images = batch["image"].to(device=DEVICE, dtype=torch.float32, memory_format=torch.channels_last)
            target_intensity = batch["target_intensity"].to(device=DEVICE, dtype=torch.float32)
            target_mask = batch["target_mask"].to(device=DEVICE, dtype=torch.bool)
            target_intensity_positive = batch["target_intensity_positive"].to(device=DEVICE, dtype=torch.float32)
            positive_scale = batch["positive_scale"].to(device=DEVICE, dtype=torch.float32)

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            inference_start = time.perf_counter()
            with torch.autocast(DEVICE.type if DEVICE.type != "mps" else "cpu", enabled=AMP):
                logits = model(images)
                if logits.shape[2:] != target_intensity.shape[2:]:
                    logits = F.interpolate(logits, size=target_intensity.shape[2:], mode="bilinear", align_corners=False)
                first_prediction = first_spot_prediction(logits, images)
                prediction = compose_spot_predictions(first_prediction, images)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - inference_start

            metric_start = time.perf_counter()
            aligned_target, aligned_mask, selected_second = align_targets_for_first_output(first_prediction, target_intensity, target_mask)
            aligned_target_positive = target_intensity_positive.clone()
            aligned_target_positive[selected_second] = target_intensity_positive[selected_second].flip(1)
            metrics = compute_metrics(prediction, aligned_target, aligned_mask, aligned_target_positive, positive_scale)
            reconstruction = prediction.sum(dim=1, keepdim=True)
            reconstruction_rmse = (reconstruction - images).square().mean(dim=(1, 2, 3)).sqrt()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            metric_seconds += time.perf_counter() - metric_start

            batch_elapsed = time.perf_counter() - batch_start
            batch_size = len(batch["sample_name"])
            batch_spots = batch_size * 2
            n_samples += batch_size
            n_spots += batch_spots

            for i, sample_name in enumerate(batch["sample_name"]):
                sample = {
                    "model": model_path.stem,
                    "model_path": str(model_path),
                    "sample_name": sample_name,
                    "selected_target_for_output": "target_1" if bool(selected_second[i].item()) else "target_0",
                    "dice_mean": float(metrics["dice"][i].mean().cpu()),
                    "iou_mean": float(metrics["iou"][i].mean().cpu()),
                    "rmse_mean": float(metrics["rmse"][i].mean().cpu()),
                    "rmse_whole_mean": float(metrics["rmse_whole"][i].mean().cpu()),
                    "nrmse_mean": float(metrics["nrmse"][i].mean().cpu()),
                    "nrmse_whole_mean": float(metrics["nrmse_whole"][i].mean().cpu()),
                    "nrmse_positive_mean": float(metrics["nrmse_positive"][i].mean().cpu()),
                    "soft_intensity_dice_mean": float(metrics["soft_intensity_dice"][i].mean().cpu()),
                    "soft_intensity_dice_whole_mean": float(metrics["soft_intensity_dice_whole"][i].mean().cpu()),
                    "integrated_intensity_abs_relative_error_mean": float(metrics["integrated_intensity_abs_relative_error"][i].mean().cpu()),
                    "spot_recognition_mean": float(metrics["spot_recognition"][i].mean().cpu()),
                    "reconstruction_rmse": float(reconstruction_rmse[i].cpu()),
                    "batch_eval_seconds": batch_elapsed,
                    "batch_seconds_per_sample": batch_elapsed / max(batch_size, 1),
                    "batch_seconds_per_spot": batch_elapsed / max(batch_spots, 1),
                    "donor_keys": batch["donor_keys"][i],
                    **batch["attrs"][i],
                }
                for bin_name in SPOT_BIN_ORDER:
                    sample[f"n_spots_{bin_name}"] = 0

                for channel in range(2):
                    recognition = float(metrics["spot_recognition"][i, channel].cpu())
                    bin_name = spot_bin(recognition)
                    sample[f"n_spots_{bin_name}"] += 1
                    spot_rows.append({
                        "model": model_path.stem,
                        "model_path": str(model_path),
                        "sample_name": sample_name,
                        "spot_channel": channel,
                        "spot_role": "direct_output" if channel == 0 else "residual_output",
                        "recognition_bin": bin_name,
                        "spot_recognition": recognition,
                        "dice": float(metrics["dice"][i, channel].cpu()),
                        "iou": float(metrics["iou"][i, channel].cpu()),
                        "rmse": float(metrics["rmse"][i, channel].cpu()),
                        "rmse_whole": float(metrics["rmse_whole"][i, channel].cpu()),
                        "nrmse": float(metrics["nrmse"][i, channel].cpu()),
                        "nrmse_whole": float(metrics["nrmse_whole"][i, channel].cpu()),
                        "nrmse_positive": float(metrics["nrmse_positive"][i, channel].cpu()),
                        "intensity_score": float(metrics["intensity_score"][i, channel].cpu()),
                        "soft_intensity_dice": float(metrics["soft_intensity_dice"][i, channel].cpu()),
                        "soft_intensity_dice_whole": float(metrics["soft_intensity_dice_whole"][i, channel].cpu()),
                        "pred_integral": float(metrics["pred_integral"][i, channel].cpu()),
                        "target_integral": float(metrics["target_integral"][i, channel].cpu()),
                        "integrated_intensity_error": float(metrics["integrated_intensity_error"][i, channel].cpu()),
                        "integrated_intensity_abs_error": float(metrics["integrated_intensity_abs_error"][i, channel].cpu()),
                        "integrated_intensity_relative_error": float(metrics["integrated_intensity_relative_error"][i, channel].cpu()),
                        "integrated_intensity_abs_relative_error": float(metrics["integrated_intensity_abs_relative_error"][i, channel].cpu()),
                        "batch_eval_seconds": batch_elapsed,
                        "batch_seconds_per_sample": batch_elapsed / max(batch_size, 1),
                        "batch_seconds_per_spot": batch_elapsed / max(batch_spots, 1),
                        "donor_keys": batch["donor_keys"][i],
                    })
                sample_rows.append(sample)

    total_seconds = time.perf_counter() - model_start
    timing = {
        "model": model_path.stem,
        "model_path": str(model_path),
        "n_batches": n_batches,
        "n_samples": n_samples,
        "n_spots": n_spots,
        "total_eval_seconds": total_seconds,
        "inference_seconds": inference_seconds,
        "metric_seconds": metric_seconds,
        "overhead_seconds": total_seconds - inference_seconds - metric_seconds,
        "seconds_per_sample": total_seconds / max(n_samples, 1),
        "seconds_per_spot": total_seconds / max(n_spots, 1),
        "samples_per_second": n_samples / max(total_seconds, 1e-12),
        "spots_per_second": n_spots / max(total_seconds, 1e-12),
    }

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(sample_rows), pd.DataFrame(spot_rows), timing


if not model_paths:
    print("No checkpoints found. Put .pth/.pt files into MODEL_PATHS or one of MODEL_SEARCH_ROOTS, then rerun from this cell.")
    results = pd.DataFrame()
    spot_results = pd.DataFrame()
    timing_results = pd.DataFrame()
else:
    evaluated = [evaluate_model(path) for path in model_paths]
    results = pd.concat([item[0] for item in evaluated], ignore_index=True)
    spot_results = pd.concat([item[1] for item in evaluated], ignore_index=True)
    timing_results = pd.DataFrame([item[2] for item in evaluated])
    results.to_csv(OUTPUT_DIR / "oneoutput_model_separation_metrics_5000.csv", index=False)
    spot_results.to_csv(OUTPUT_DIR / "oneoutput_spot_separation_metrics_5000.csv", index=False)
    timing_results.to_csv(OUTPUT_DIR / "oneoutput_evaluation_timing_5000.csv", index=False)
    display(results.head())
    display(spot_results.head())
    display(timing_results)

## Dataset-level metrics

In [ ]:
def dataset_label(experiment):
    text = str(experiment).replace(".h5", "").replace("_logtif", "")
    if text in {"Al_segvol", "Al"}:
        return "Al"
    if text.startswith("Al_deformed_LoG"):
        return "Al_deformed_LoG"
    if text.startswith("Al_big_grains"):
        return "Al_big_grains"
    if text.startswith("Al_small_grains"):
        return "Al_small_grains"
    if text.startswith("Cu"):
        return "Cu"
    if text.startswith("IN718_twins"):
        return "IN718_twins"
    if text.startswith("Iron_deformed"):
        return "Iron_deformed"
    if text.startswith("Iron"):
        return "Iron"
    if text.startswith("Ti7Al"):
        return "Ti7Al"
    return text


if not results.empty and "experiment" in results.columns:
    results["dataset"] = results["experiment"].map(dataset_label)
    dataset_summary = (
        results.groupby(["model", "dataset"])
        .agg(
            n_samples=("sample_name", "count"),
            dice_mean=("dice_mean", "mean"),
            iou_mean=("iou_mean", "mean"),
            rmse_mean=("rmse_mean", "mean"),
            rmse_whole_mean=("rmse_whole_mean", "mean"),
            nrmse_mean=("nrmse_mean", "mean"),
            nrmse_whole_mean=("nrmse_whole_mean", "mean"),
            nrmse_positive_mean=("nrmse_positive_mean", "mean"),
            soft_intensity_dice_mean=("soft_intensity_dice_mean", "mean"),
            soft_intensity_dice_whole_mean=("soft_intensity_dice_whole_mean", "mean"),
            integrated_intensity_abs_relative_error_mean=("integrated_intensity_abs_relative_error_mean", "mean"),
            spot_recognition_mean=("spot_recognition_mean", "mean"),
            spot_recognition_p10=("spot_recognition_mean", lambda x: x.quantile(0.10)),
            reconstruction_rmse=("reconstruction_rmse", "mean"),
        )
        .sort_values(["model", "dice_mean"], ascending=[True, False])
    )
    dataset_summary.to_csv(OUTPUT_DIR / "oneoutput_dataset_separation_metrics_5000.csv")
    display(dataset_summary)
else:
    print("No experiment column found; cannot compute dataset-level metrics.")

## Spot counter and model summary

In [ ]:
if not results.empty:
    summary = (
        results.groupby("model")
        .agg(
            n_samples=("sample_name", "count"),
            dice_mean=("dice_mean", "mean"),
            iou_mean=("iou_mean", "mean"),
            rmse_mean=("rmse_mean", "mean"),
            rmse_whole_mean=("rmse_whole_mean", "mean"),
            nrmse_mean=("nrmse_mean", "mean"),
            nrmse_whole_mean=("nrmse_whole_mean", "mean"),
            nrmse_positive_mean=("nrmse_positive_mean", "mean"),
            soft_intensity_dice_mean=("soft_intensity_dice_mean", "mean"),
            soft_intensity_dice_whole_mean=("soft_intensity_dice_whole_mean", "mean"),
            integrated_intensity_abs_relative_error_mean=("integrated_intensity_abs_relative_error_mean", "mean"),
            spot_recognition_mean=("spot_recognition_mean", "mean"),
            reconstruction_rmse=("reconstruction_rmse", "mean"),
        )
        .sort_values("spot_recognition_mean", ascending=False)
    )
    if not timing_results.empty:
        timing_cols = ["model", "total_eval_seconds", "seconds_per_sample", "seconds_per_spot", "samples_per_second", "spots_per_second"]
        summary = summary.join(timing_results[timing_cols].set_index("model"), how="left")
    summary.to_csv(OUTPUT_DIR / "oneoutput_model_summary_5000.csv")
    display(summary)

if not spot_results.empty:
    spot_results["recognition_bin"] = pd.Categorical(spot_results["recognition_bin"], categories=SPOT_BIN_ORDER, ordered=True)
    spot_counter = (
        spot_results.groupby(["model", "recognition_bin"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=SPOT_BIN_ORDER, fill_value=0)
    )
    spot_counter["total_spots"] = spot_counter.sum(axis=1)
    for bin_name in SPOT_BIN_ORDER:
        spot_counter[f"{bin_name}_fraction"] = spot_counter[bin_name] / spot_counter["total_spots"].clip(lower=1)
    spot_counter.to_csv(OUTPUT_DIR / "oneoutput_spot_counter_5000.csv")
    display(spot_counter)

## Donor-level metrics

In [ ]:
def explode_donors(df):
    rows = []
    for row in df.to_dict("records"):
        donor_keys = row.get("donor_keys") or []
        for donor_key in donor_keys:
            new_row = row.copy()
            new_row["donor_key"] = donor_key
            rows.append(new_row)
    return pd.DataFrame(rows)


if not results.empty:
    donor_results = explode_donors(results)
    if donor_results.empty:
        print("No donor metadata found in this HDF5 file.")
    else:
        donor_summary = (
            donor_results.groupby(["model", "donor_key"])
            .agg(
                n=("sample_name", "count"),
                dice_mean=("dice_mean", "mean"),
                iou_mean=("iou_mean", "mean"),
                rmse_mean=("rmse_mean", "mean"),
                soft_intensity_dice_mean=("soft_intensity_dice_mean", "mean"),
            soft_intensity_dice_whole_mean=("soft_intensity_dice_whole_mean", "mean"),
                spot_recognition_mean=("spot_recognition_mean", "mean"),
                spot_recognition_p10=("spot_recognition_mean", lambda x: x.quantile(0.10)),
            )
            .query("n >= 3")
            .sort_values(["model", "spot_recognition_mean"], ascending=[True, True])
        )
        donor_summary.to_csv(OUTPUT_DIR / "oneoutput_donor_separation_metrics_5000.csv")
        display(donor_summary.head(20))
        print("Best donors:")
        display(donor_summary.sort_values("spot_recognition_mean", ascending=False).head(20))

## Worst examples for visual inspection

In [ ]:
def plot_sample_prediction(model_path, sample_name):
    model = load_oneoutput_model(model_path)
    single = H5OneOutputSubsetDataset(DATA_PATH, [sample_name])
    batch = collate_batch([single[0]])
    image = batch["image"].to(DEVICE, dtype=torch.float32)
    target_intensity = batch["target_intensity"].to(DEVICE, dtype=torch.float32)
    target_mask = batch["target_mask"].to(DEVICE, dtype=torch.bool)

    with torch.no_grad():
        logits = model(image)
        if logits.shape[2:] != target_intensity.shape[2:]:
            logits = F.interpolate(logits, size=target_intensity.shape[2:], mode="bilinear", align_corners=False)
        first = first_spot_prediction(logits, image)
        pred = compose_spot_predictions(first, image)
        target_intensity, target_mask, _ = align_targets_for_first_output(first, target_intensity, target_mask)

    image_np = image[0, 0].cpu().numpy()
    target_np = target_intensity[0].cpu().numpy()
    pred_np = pred[0].cpu().numpy()

    fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
    panels = [
        (image_np, "input", "gray"),
        (target_np[0], "target selected spot", "gray"),
        (pred_np[0], "pred direct spot", "gray"),
        (np.abs(pred_np[0] - target_np[0]), "direct error", "magma"),
        (target_np.sum(axis=0), "target sum", "gray"),
        (target_np[1], "target residual spot", "gray"),
        (pred_np[1], "pred residual spot", "gray"),
        (np.abs(pred_np[1] - target_np[1]), "residual error", "magma"),
    ]
    for ax, (data, title, cmap) in zip(axes.flat, panels):
        ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{Path(model_path).name} | {sample_name}")
    plt.show()
    del model


if not results.empty:
    worst = results.sort_values("spot_recognition_mean").head(10)[[
        "model", "model_path", "sample_name", "selected_target_for_output", "dice_mean", "iou_mean", "rmse_mean", "soft_intensity_dice_mean", "spot_recognition_mean", "donor_keys"
    ]]
    display(worst)
    first = worst.iloc[0]
    plot_sample_prediction(Path(first["model_path"]), first["sample_name"])